# 01 – Data Loading & Cleaning
Project: Olist E-commerce Analytics
Goal: inspect each raw table, document every issue found, fix what needs fixing, and save clean files to `data/processed/`.

Decision log convention: every finding ends in either an **Action** or "No action needed, because…".

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path('.').resolve().parent
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROCESSED_DIR = BASE_DIR / 'data' / 'processed'

print(pd.__version__)
print(RAW_DIR.exists(), PROCESSED_DIR.exists())

3.0.6
True True


## Customers table
### Task 1: Load the raw file
Load `olist_customers_dataset.csv` into `customers_df` (raw, never modified) and work on `customers` (a copy).

In [9]:
customers_df = pd.read_csv(RAW_DIR / 'olist_customers_dataset.csv')
customers = customers_df.copy()
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


**Completed:** loaded the raw file into `customers_df`, working copy is `customers`.
**Result:** 5 columns: customer_id, customer_unique_id, customer_zip_code_prefix, customer_city, customer_state.

In [10]:
print(customers.shape)
customers.info()

(99441, 5)
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               99441 non-null  str  
 1   customer_unique_id        99441 non-null  str  
 2   customer_zip_code_prefix  99441 non-null  int64
 3   customer_city             99441 non-null  str  
 4   customer_state            99441 non-null  str  
dtypes: int64(1), str(4)
memory usage: 3.8 MB


**Completed:** checked shape and dtypes.
**Result:** 99,441 rows × 5 columns. Four columns are text, one (`customer_zip_code_prefix`) is int64.
**Finding:** the zip prefix should be text, not a number (handled in Task 8).

### Task 3: Missing values
Count nulls in every column.

In [11]:
customers.isnull().sum()

customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

**Completed:** counted nulls per column.
**Result:** 0 nulls in all 5 columns.
**Action:** none needed, because the table is complete.

### Task 4: Duplicate rows
Check for fully duplicated rows.

In [12]:
print('Duplicate rows:', customers.duplicated().sum())

Duplicate rows: 0


**Completed:** checked duplicates.
**Result:** 0 duplicate rows.
**Action:** none needed, because there is nothing to remove.

### Task 5: `customer_id` vs `customer_unique_id`
Find out which column identifies a real person, and how many repeat customers exist.

In [ ]:
n_ids = customers['customer_id'].nunique()
n_people = customers['customer_unique_id'].nunique()

print('customer_id unique:       ', n_ids)
print('customer_unique_id unique:', n_people)
print('Rows from returning customers:', n_ids - n_people) #repeat customers

# How many customer_id rows does each real person have?
customers['customer_unique_id'].value_counts().value_counts().sort_index()

customer_id unique:        99441
customer_unique_id unique: 96096
Rows from returning customers: 3345


count
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

**Completed:** compared the two ID columns.
**Result:** 99,441 `customer_id` values but only 96,096 `customer_unique_id` values, so 3,345 rows belong to returning customers.
**Finding:** `customer_id` is created per order; `customer_unique_id` identifies the real person.
**Action:** use `customer_unique_id` for every customer count, repeat-purchase and retention analysis in this project.

### Task 6: Customer distribution by state
Check state codes are valid and see where customers are concentrated.

In [14]:
state_counts = customers['customer_state'].value_counts()
state_share = (customers['customer_state'].value_counts(normalize=True) * 100).round(2)

print('Unique states:', len(state_counts))
print(pd.concat([state_counts, state_share], axis=1, keys=['customers', 'share_%']).head(5))
print('Top 3 share:', state_share.head(3).sum().round(1), '%')

Unique states: 27
                customers  share_%
customer_state                    
SP                  41746    41.98
RJ                  12852    12.92
MG                  11635    11.70
RS                   5466     5.50
PR                   5045     5.07
Top 3 share: 66.6 %


**Completed:** counted customers per state.
**Result:** 27 unique states (matches Brazil's 26 states + Federal District). SP 41,746 (42.0%), RJ 12,852 (12.9%), MG 11,635 (11.7%); top 3 states ≈ 66.6% of customers.
**Insight:** demand is heavily concentrated in the Southeast. Revisit this in the dashboard and in the delivery analysis.
**Action:** none needed.